In [1]:
import polars as pl

In [2]:
articles_path='../data/articles.parquet'
transaction_path='../data/transactions.parquet'
customers_path='../data/customers.parquet'

In [3]:
articles=pl.read_parquet(articles_path)
transactions=pl.read_parquet(transaction_path)
customers=pl.read_parquet(customers_path)

In [4]:
transactions.head()

customer_id,article_id,price,sales_channel_id,time
str,i64,f64,u8,f64
"""000058a12d5b43e67d225668fa1f8d…",663713001,0.050831,2,1.5374e9
"""000058a12d5b43e67d225668fa1f8d…",541518023,0.030492,2,1.5374e9
"""00007d2de826758b65a93dd24ce629…",505221004,0.015237,2,1.5374e9
"""00007d2de826758b65a93dd24ce629…",685687003,0.016932,2,1.5374e9
"""00007d2de826758b65a93dd24ce629…",685687004,0.016932,2,1.5374e9


In [5]:
articles.head()

article_id,product_code,product_type_no,graphical_appearance_no,colour_group_code,perceived_colour_value_id,perceived_colour_master_id,department_no,index_code,index_group_no,section_no,garment_group_no
i64,i64,i64,i64,i64,i64,i64,i64,str,i64,i64,i64
108775015,108775,253,1010016,9,4,5,1676,"""A""",1,16,1002
108775044,108775,253,1010016,10,3,9,1676,"""A""",1,16,1002
108775051,108775,253,1010017,11,1,9,1676,"""A""",1,16,1002
110065001,110065,306,1010016,9,4,5,1339,"""B""",1,61,1017
110065002,110065,306,1010016,10,3,9,1339,"""B""",1,61,1017


In [6]:
customers.head()

customer_id,FN,Active,fashion_news_frequency_Monthly,fashion_news_frequency_NONE,fashion_news_frequency_Regularly,club_member_status_ACTIVE,club_member_status_LEFT CLUB,club_member_status_PRE-CREATE,age_0,age_1,age_2
str,u8,u8,bool,bool,bool,bool,bool,bool,bool,bool,bool
"""00000dbacae5abe5e23885899a1fa4…",0,0,false,true,false,true,false,false,false,true,false
"""0000423b00ade91418cceaf3b26c6a…",0,0,false,true,false,true,false,false,true,false,false
"""000058a12d5b43e67d225668fa1f8d…",0,0,false,true,false,true,false,false,true,false,false
"""00005ca1c9ed5f5146b52ac8639a40…",0,0,false,true,false,true,false,false,false,true,false
"""00006413d8573cd20ed7128e53b7b1…",1,1,false,false,true,true,false,false,false,true,false


In [7]:
transactions=transactions.join(
    articles.select(['article_id','product_type_no']),
    on=['article_id'],
    how='left'
)
transactions.columns

['customer_id',
 'article_id',
 'price',
 'sales_channel_id',
 'time',
 'product_type_no']

In [8]:
res=transactions.group_by('customer_id').agg(pl.col('time').max().alias('time_max'),pl.col('time').min().alias('time_min'))
customers=customers.join(res,on=['customer_id'],how='left')
customers.head()

customer_id,FN,Active,fashion_news_frequency_Monthly,fashion_news_frequency_NONE,fashion_news_frequency_Regularly,club_member_status_ACTIVE,club_member_status_LEFT CLUB,club_member_status_PRE-CREATE,age_0,age_1,age_2,time_max,time_min
str,u8,u8,bool,bool,bool,bool,bool,bool,bool,bool,bool,f64,f64
"""00000dbacae5abe5e23885899a1fa4…",0,0,false,true,false,true,false,false,false,true,false,1.5993e9,1.5459e9
"""0000423b00ade91418cceaf3b26c6a…",0,0,false,true,false,true,false,false,true,false,false,1.5942e9,1.5375e9
"""000058a12d5b43e67d225668fa1f8d…",0,0,false,true,false,true,false,false,true,false,false,1.6001e9,1.5374e9
"""00005ca1c9ed5f5146b52ac8639a40…",0,0,false,true,false,true,false,false,false,true,false,1.5600e9,1.5600e9
"""00006413d8573cd20ed7128e53b7b1…",1,1,false,false,true,true,false,false,false,true,false,1.5972e9,1.5393e9


In [9]:
transactions.sort(['customer_id','time'])

customer_id,article_id,price,sales_channel_id,time,product_type_no
str,i64,f64,u8,f64,i64
"""00000dbacae5abe5e23885899a1fa4…",627759010,0.030492,1,1.5459e9,262
"""00000dbacae5abe5e23885899a1fa4…",176209023,0.035576,1,1.5459e9,308
"""00000dbacae5abe5e23885899a1fa4…",625548001,0.044051,1,1.5459e9,262
"""00000dbacae5abe5e23885899a1fa4…",697138006,0.010153,2,1.5568e9,267
"""00000dbacae5abe5e23885899a1fa4…",568601006,0.050831,2,1.5587e9,264
…,…,…,…,…,…
"""ffffd7744cebcf3aca44ae7049d2a9…",840360003,0.013542,2,1.5864e9,255
"""ffffd7744cebcf3aca44ae7049d2a9…",866755002,0.043203,2,1.5864e9,265
"""ffffd7744cebcf3aca44ae7049d2a9…",866755002,0.050831,2,1.5878e9,265


In [10]:
def feature_cnt(data: pl.DataFrame, transactions: pl.DataFrame) -> pl.DataFrame:
    DAY = 86400
    WINDOWS = {
        7: 7 * DAY,
        30: 30 * DAY,
        90: 90 * DAY,
    }

    now = transactions.select(pl.col("time").max()).item()

    for days, window in WINDOWS.items():
        cnt_col = f"cnt_{days}"

        # 防止重复调用时列冲突
        if cnt_col in data.columns:
            data = data.drop(cnt_col)

        res = (
            transactions
            .filter(pl.col("time") >= now - window)
            .with_columns(
                (1.0 / (1.0 + (now - pl.col("time")) / DAY)).alias("time_weight")
            )
            .group_by(["customer_id", "article_id"])
            .agg(pl.col("time_weight").sum().alias(cnt_col))
        )

        data = (
            data
            .join(res, on=["customer_id", "article_id"], how="left")
            .with_columns(pl.col(cnt_col).fill_null(0.0))
        )

    return data